# Domain F: Recurrent Neural Network Modeling

This notebook addresses two questions:
- **F1:** Can a task-trained RNN (Dale's law constrained, E/I ratio matching data) reproduce cell-type-specific tuning and connectivity?
- **F2:** Can a data-driven RNN trained to predict population ΔF/F learn biologically meaningful representations?

**Dependencies:** `torch` (PyTorch). Install via `pip install torch`.

**Note:** Both models use the actual stimulus parameters from the dataset but are trained on synthetic or simplified tasks. Sections requiring additional data or long training are marked.

In [ ]:
import sys
sys.path.insert(0, '.')

import numpy as np
import pandas as pd
import zarr
import matplotlib.pyplot as plt
import seaborn as sns
from types import SimpleNamespace
from scipy.stats import pearsonr, spearmanr
import warnings
warnings.filterwarnings('ignore')

try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    TORCH_AVAILABLE = True
    print(f"PyTorch version: {torch.__version__}")
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}")
except ImportError:
    TORCH_AVAILABLE = False
    print("⚠️ PyTorch not installed. Install via: pip install torch")

from functions.data_loading import load_mouse_zarr, load_zarr_10hz
from functions.analysis import linear_CKA
from functions.models import DalesRNN, PredictiveRNN, TemporalRNN

sns.set_context('talk')
sns.set_style('whitegrid')

# ══════════════════════════════════════════════════════════════════════
# Load data from zarr multimodal stores
# ══════════════════════════════════════════════════════════════════════
ZARR_DIR = 'multimodal_data'
MOUSE_IDS = ['778174', '786297', '797371']
SESSIONS = ['session_1', 'session_2', 'session_3']

adata_list = [load_mouse_zarr(mid, zarr_dir=ZARR_DIR) for mid in MOUSE_IDS]

obs = pd.concat([a.obs for a in adata_list], ignore_index=True)
X = np.vstack([a.X for a in adata_list])
var = adata_list[0].var.copy()

SUBCLASS_ORDER = ['007 L2/3 IT CTX Glut', '006 L4/5 IT CTX Glut', '022 L5 ET CTX Glut',
                  '052 Pvalb Gaba', '053 Sst Gaba', '046 Vip Gaba', '049 Lamp5 Gaba']
SUBCLASS_COLORS = {'007 L2/3 IT CTX Glut': '#1f77b4', '006 L4/5 IT CTX Glut': '#2ca02c',
                   '022 L5 ET CTX Glut': '#9467bd', '052 Pvalb Gaba': '#d62728',
                   '053 Sst Gaba': '#ff7f0e', '046 Vip Gaba': '#e377c2', '049 Lamp5 Gaba': '#8c564b'}
SUBCLASS_SHORT = {'007 L2/3 IT CTX Glut': 'L2/3 IT', '006 L4/5 IT CTX Glut': 'L4/5 IT',
                  '022 L5 ET CTX Glut': 'L5 ET', '052 Pvalb Gaba': 'Pvalb',
                  '053 Sst Gaba': 'Sst', '046 Vip Gaba': 'Vip', '049 Lamp5 Gaba': 'Lamp5'}
present_subclasses = [s for s in SUBCLASS_ORDER if s in obs['subclass_name'].unique()]

# Backward-compatible adata object for downstream cells
adata = SimpleNamespace(X=X, obs=obs, var=var, n_obs=len(obs), n_vars=X.shape[1])

orientations = np.array([0, 45, 90, 135, 180, 225, 270, 315])
contrasts = np.array([0.05, 0.1, 0.2, 0.4, 0.8])
tfs = np.array([1, 2, 4, 8, 15])

# ── Data-derived E/I ratio ──
sc_counts = obs['subclass_name'].value_counts()
exc_count = sum(sc_counts.get(s, 0) for s in present_subclasses if 'Glut' in s)
inh_count = sum(sc_counts.get(s, 0) for s in present_subclasses if 'Gaba' in s)
print(f"\nExcitatory: {exc_count} ({exc_count/len(obs):.1%}), Inhibitory: {inh_count} ({inh_count/len(obs):.1%})")
print(f"E/I ratio: {exc_count/max(inh_count,1):.1f}:1")

## F3: Temporal Trajectory RNN (10 Hz ΔF/F)

Using the 10 Hz trial-resolved traces, we train an RNN to predict the **full temporal trajectory** of population ΔF/F within each trial (41 time steps from -1s to +3s), rather than just the trial-averaged scalar. This enables:
- Comparison of temporal dynamics between model and real neurons
- Assessing whether the RNN learns transient/sustained response profiles
- CKA comparison at each time point within the trial

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# F3.1  Load 10 Hz data & train RNN on full temporal trajectories
# ══════════════════════════════════════════════════════════════════════

if TORCH_AVAILABLE:
    # Load from the largest mouse
    mouse_f3 = obs['mouse_id'].value_counts().idxmax()
    pk = load_zarr_10hz(mouse_f3)
    
    dff_10hz = pk['dff']              # (n_trials, 41, n_cells)
    run_10hz = pk['running'][:, :, 0] # (n_trials, 41) speed
    time_rel_f3 = pk['time_rel']      # (41,)
    ti_f3 = pk['trial_info']
    uids_f3 = pk['unique_ids']
    
    n_trials_f3, n_tp_f3, n_cells_f3 = dff_10hz.shape
    
    # Non-gray grating trials
    non_gray_f3 = ~ti_f3['gray_screen'].astype(bool)
    grating_idx_f3 = np.where(non_gray_f3.values)[0]
    n_gt_f3 = len(grating_idx_f3)
    
    # Subsample cells for tractability
    np.random.seed(42)
    n_target_f3 = min(80, n_cells_f3)
    target_cells_f3 = np.sort(np.random.choice(n_cells_f3, n_target_f3, replace=False))
    
    # Match subclasses
    obs_mouse_f3 = obs[obs['mouse_id'] == mouse_f3]
    target_sc_f3 = []
    for ci in target_cells_f3:
        uid = uids_f3[ci]
        m = obs_mouse_f3[obs_mouse_f3['unique_id'] == uid]
        target_sc_f3.append(m.iloc[0]['subclass_name'] if len(m) > 0 else 'Unknown')
    target_sc_f3 = np.array(target_sc_f3)
    
    # ── Build input/output for RNN ──
    ori_f3 = ti_f3.loc[grating_idx_f3, 'orientation'].values.astype(float)
    contrast_f3 = ti_f3.loc[grating_idx_f3, 'contrast'].values.astype(float)
    tf_f3 = ti_f3.loc[grating_idx_f3, 'temporal_frequency'].values.astype(float)
    
    X_input_f3 = np.zeros((n_gt_f3, n_tp_f3, 6))
    for tr in range(n_gt_f3):
        ori_rad = np.deg2rad(ori_f3[tr])
        X_input_f3[tr, :, 0] = np.cos(ori_rad)
        X_input_f3[tr, :, 1] = np.sin(ori_rad)
        X_input_f3[tr, :, 2] = contrast_f3[tr] / 0.8
        X_input_f3[tr, :, 3] = np.log2(tf_f3[tr]) / 4
        X_input_f3[tr, :, 4] = np.clip(run_10hz[grating_idx_f3[tr]] / 30, -1, 2)
        X_input_f3[tr, :, 5] = time_rel_f3 / 3.0
    
    Y_output_f3 = dff_10hz[grating_idx_f3][:, :, target_cells_f3]
    
    # Train/test split
    perm_f3 = np.random.permutation(n_gt_f3)
    n_train_f3 = int(0.8 * n_gt_f3)
    train_f3 = perm_f3[:n_train_f3]
    test_f3 = perm_f3[n_train_f3:]
    
    X_train_f3 = torch.FloatTensor(X_input_f3[train_f3]).to(device)
    X_test_f3 = torch.FloatTensor(X_input_f3[test_f3]).to(device)
    Y_train_f3 = torch.FloatTensor(Y_output_f3[train_f3]).to(device)
    Y_test_f3 = torch.FloatTensor(Y_output_f3[test_f3]).to(device)
    
    # Initialize temporal RNN (imported from functions.models)
    temporal_model = TemporalRNN(n_input=6, n_hidden=256, n_output=n_target_f3).to(device)
    opt_f3 = optim.Adam(temporal_model.parameters(), lr=1e-3)
    
    # Training
    batch_size_f3 = 128
    n_epochs_f3 = 150
    losses_f3 = []
    
    print(f"Training temporal trajectory RNN: {n_train_f3} trials, 41 timepoints, {n_target_f3} cells")
    for epoch in range(n_epochs_f3):
        temporal_model.train()
        epoch_loss = 0
        idx_shuffled = np.random.permutation(n_train_f3)
        
        for start in range(0, n_train_f3, batch_size_f3):
            batch = idx_shuffled[start:start+batch_size_f3]
            pred, _ = temporal_model(X_train_f3[batch])
            loss = nn.MSELoss()(pred, Y_train_f3[batch])
            
            opt_f3.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(temporal_model.parameters(), 1.0)
            opt_f3.step()
            epoch_loss += loss.item()
        
        losses_f3.append(epoch_loss / max(1, n_train_f3 // batch_size_f3))
        
        if (epoch + 1) % 30 == 0:
            temporal_model.eval()
            with torch.no_grad():
                pred_test_f3, _ = temporal_model(X_test_f3)
                test_loss_f3 = nn.MSELoss()(pred_test_f3, Y_test_f3).item()
            print(f"  Epoch {epoch+1}: train={losses_f3[-1]:.4f}, test={test_loss_f3:.4f}")
    
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(losses_f3, 'b-', linewidth=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.set_title('F3: Training Loss — Temporal Trajectory RNN', fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ Skipping — PyTorch not available.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# F3.2  Compare temporal dynamics: predicted vs real PSTH, time-resolved CKA
# ══════════════════════════════════════════════════════════════════════

if TORCH_AVAILABLE:
    from sklearn.decomposition import PCA
    
    temporal_model.eval()
    with torch.no_grad():
        pred_f3, hidden_f3 = temporal_model(X_test_f3)
    
    pred_f3_np = pred_f3.cpu().numpy()      # (n_test, 41, n_target)
    hidden_f3_np = hidden_f3.cpu().numpy()  # (n_test, 41, n_hidden)
    actual_f3_np = Y_test_f3.cpu().numpy()  # (n_test, 41, n_target)
    n_test_f3 = actual_f3_np.shape[0]
    
    # ── 1. Compare mean predicted vs real PSTH (averaged across test trials) ──
    mean_pred_psth = np.nanmean(pred_f3_np, axis=0)   # (41, n_target)
    mean_real_psth = np.nanmean(actual_f3_np, axis=0)  # (41, n_target)
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    # Per-subclass mean PSTH: real vs predicted
    present_sc_f3 = [s for s in SUBCLASS_ORDER if s in target_sc_f3]
    for ax, sc in zip(axes[0], present_sc_f3[:3]):
        sc_mask = target_sc_f3 == sc
        if sc_mask.sum() < 2:
            continue
        real_mean = np.nanmean(mean_real_psth[:, sc_mask], axis=1)
        pred_mean = np.nanmean(mean_pred_psth[:, sc_mask], axis=1)
        ax.plot(time_rel_f3, real_mean, color=SUBCLASS_COLORS[sc], linewidth=2, label='Real')
        ax.plot(time_rel_f3, pred_mean, color=SUBCLASS_COLORS[sc], linewidth=2, ls='--', label='Predicted')
        ax.axvline(0, color='k', ls='--', alpha=0.3)
        ax.axvline(2.0, color='k', ls=':', alpha=0.3)
        ax.set_title(f'{SUBCLASS_SHORT[sc]} (n={sc_mask.sum()})',
                     color=SUBCLASS_COLORS[sc], fontweight='bold')
        ax.set_xlabel('Time (s)')
        ax.legend(fontsize=8)
    axes[0, 0].set_ylabel('ΔF/F')
    
    # Handle overflow subclasses
    for ax, sc in zip(axes[1][:len(present_sc_f3[3:])], present_sc_f3[3:]):
        sc_mask = target_sc_f3 == sc
        if sc_mask.sum() < 2:
            continue
        real_mean = np.nanmean(mean_real_psth[:, sc_mask], axis=1)
        pred_mean = np.nanmean(mean_pred_psth[:, sc_mask], axis=1)
        ax.plot(time_rel_f3, real_mean, color=SUBCLASS_COLORS[sc], linewidth=2, label='Real')
        ax.plot(time_rel_f3, pred_mean, color=SUBCLASS_COLORS[sc], linewidth=2, ls='--', label='Predicted')
        ax.axvline(0, color='k', ls='--', alpha=0.3)
        ax.set_title(f'{SUBCLASS_SHORT[sc]} (n={sc_mask.sum()})',
                     color=SUBCLASS_COLORS[sc], fontweight='bold')
        ax.legend(fontsize=8)
    
    # ── 2. Time-resolved CKA ──
    cka_per_tp = np.zeros(n_tp_f3)
    n_sub_cka = min(300, n_test_f3)
    cka_idx = np.random.choice(n_test_f3, n_sub_cka, replace=False)
    
    for tp in range(n_tp_f3):
        real_tp = actual_f3_np[cka_idx, tp, :]   # (n_sub, n_cells)
        hidden_tp = hidden_f3_np[cka_idx, tp, :]  # (n_sub, n_hidden)
        cka_per_tp[tp] = linear_CKA(real_tp, hidden_tp)
    
    ax = axes[1, -2] if len(present_sc_f3) <= 3 else axes[1, len(present_sc_f3[3:])]
    ax.plot(time_rel_f3, cka_per_tp, 'b-', linewidth=2, marker='o', markersize=3)
    ax.axvline(0, color='k', ls='--', alpha=0.4)
    ax.axvline(2.0, color='k', ls=':', alpha=0.4)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('CKA')
    ax.set_title('F3: Time-Resolved CKA\n(Real vs RNN Hidden)', fontweight='bold')
    ax.set_ylim(0, max(0.5, cka_per_tp.max() * 1.2))
    
    # ── 3. Per-cell temporal prediction quality ──
    per_cell_temporal_r = np.zeros(n_target_f3)
    for ci in range(n_target_f3):
        real_flat = actual_f3_np[:, :, ci].ravel()
        pred_flat = pred_f3_np[:, :, ci].ravel()
        valid = ~np.isnan(real_flat) & ~np.isnan(pred_flat)
        if valid.sum() > 10:
            per_cell_temporal_r[ci] = np.corrcoef(real_flat[valid], pred_flat[valid])[0, 1]
    
    ax_last = axes.flat[-1]
    for sc in present_sc_f3:
        sc_mask = target_sc_f3 == sc
        vals = per_cell_temporal_r[sc_mask]
        if len(vals) >= 2:
            ax_last.scatter([SUBCLASS_SHORT[sc]] * len(vals), vals, alpha=0.5, s=20,
                           color=SUBCLASS_COLORS[sc])
            ax_last.scatter(SUBCLASS_SHORT[sc], np.mean(vals), s=100, color=SUBCLASS_COLORS[sc],
                           edgecolors='k', zorder=5, marker='D')
    ax_last.axhline(0, color='k', ls='--', alpha=0.4)
    ax_last.set_ylabel('Pearson r (temporal)')
    ax_last.set_title('F3: Per-Cell Temporal Prediction', fontweight='bold')
    ax_last.tick_params(axis='x', rotation=45)
    
    # Hide unused axes
    for ax in axes.flat:
        if not ax.has_data():
            ax.set_visible(False)
    
    plt.suptitle('F3: Temporal Trajectory RNN Analysis', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print(f"\nMedian per-cell temporal r: {np.median(per_cell_temporal_r):.3f}")
    print(f"Mean time-resolved CKA: {np.mean(cka_per_tp):.4f}")
    print(f"Peak CKA at t={time_rel_f3[np.argmax(cka_per_tp)]:.1f}s: {np.max(cka_per_tp):.4f}")
else:
    print("⚠️ Skipping — PyTorch not available.")